# Global Corporate Counterparty Rating (GCCR)
## Notebook 02: Probability of Default (PD) Modeling & Scorecard Calibration

In this notebook, we train institutional credit scorecards (Logistic Regression and XGBoost), execute Platt Scaling (Logistic Sigmoidal Calibration), and inspect scorecard feature weights.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("..")

from src.data.edgar_loader import load_sec_financials
from src.features.financial_ratios import compute_financial_ratios
from src.models.pd_engine import train_and_calibrate_pd_model

print("Setup completed successfully.")


### 1. Ingest Features & Prepare Training Cohort

In [ ]:
df_raw = load_sec_financials("../data/raw/edgar_sec_financials.csv")
df_features = compute_financial_ratios(df_raw)
print(f"Features engineered: {len(df_features)} observations")


### 2. Train Scorecard with Platt Sigmoidal Calibration

In [ ]:
df_scored, pd_model = train_and_calibrate_pd_model(
    df_features, 
    algorithm="logistic_regression",
    calibration_method="sigmoid"
)

print(f"Calibrated 1-year PD summary:")
print(df_scored[['pd_uncalibrated', 'pd_calibrated']].describe().round(4))


### 3. Feature Coefficients & Scorecard Interpretability

In [ ]:
weights = pd_model.get_feature_weights()
weights_df = pd.DataFrame(list(weights.items()), columns=['Feature', 'Coefficient']).sort_values(by='Coefficient')

print("Scorecard Risk Drivers:")
print(weights_df.to_string(index=False))
